# The Information Bottleneck

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 05.04 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/05_Information_Theory/05_information_bottleneck.ipynb)

---

## 🎯 Learning Objective

Learn the information bottleneck principle as a compression-prediction tradeoff, and see how it offers a theory of deep learning generalization.

---

## 📐 Theory

Suppose you want to compress data $X$ into a representation $T$, but not just *any*
compression — one that keeps everything relevant to predicting some target $Y$, while throwing
away everything else. This is a genuinely different goal from ordinary compression (which just
minimizes size), and the **information bottleneck (IB)** principle (Tishby, Pereira & Bialek,
1999) makes it a precise optimization problem using the machinery of `05.01`–`05.04`.

### The tradeoff, stated with mutual information

Let $X$ be the input, $Y$ the target you ultimately care about, and $T$ a (possibly stochastic)
representation of $X$ — think of $T$ as a hidden layer's activations in a neural network. Two
quantities compete:

- $I(X;T)$ — how much information $T$ retains about the raw input $X$. Large $I(X;T)$ means
  $T$ is a rich, high-fidelity (but possibly bloated, overfit-prone) encoding of $X$.
- $I(T;Y)$ — how much information $T$ carries about the target $Y$, i.e. how *useful* $T$ is
  for the task. By the data processing inequality (`05.04`), since $Y \to X \to T$ forms a
  Markov chain in the sense that $T$ only sees $X$, $I(T;Y) \le I(X;Y)$ — no representation can
  ever be more informative about $Y$ than the raw input itself was.

The information bottleneck asks for the representation $T$ that solves

$$\min_{p(t|x)} \; I(X;T) - \beta\, I(T;Y)$$

$\beta > 0$ is a Lagrange multiplier trading off the two goals. Small $\beta$ prioritizes
aggressive compression ($T$ becomes tiny, minimal $I(X;T)$, potentially useless for $Y$); large
$\beta$ prioritizes prediction ($T$ is allowed to retain as much of $X$ as needed to maximize
$I(T;Y)$, up to the ceiling $I(X;Y)$). Sweeping $\beta$ from $0$ to $\infty$ traces out the
**information bottleneck curve** — the best achievable $I(T;Y)$ for every possible compression
budget $I(X;T)$, directly analogous to a Pareto frontier trading off two competing objectives.

### Connection to rate-distortion theory

This is a direct descendant of classical **rate-distortion theory**: given a "rate" budget (bits
used to encode $X$ as $T$, i.e. $I(X;T)$), what's the minimum achievable "distortion"? The IB
framework replaces a generic distortion metric with a specifically information-theoretic one —
distortion is measured as *lost predictive information about $Y$*, i.e. $I(X;Y) - I(T;Y)$,
rather than, say, squared reconstruction error. The **Blahut-Arimoto algorithm**, originally
developed for rate-distortion problems, adapts directly to solving the IB objective by
alternating between updating the stochastic encoder $p(t|x)$ and the induced marginals — we
implement this alternating procedure from scratch below.

### The information bottleneck as a theory of deep learning

Tishby & Zaslavsky (2015) proposed that training a deep network is itself an implicit
information bottleneck process: each layer $T_\ell$ represents the input $X$, and training tries
to maximize $I(T_\ell; Y)$ while — the more controversial claim — a "compression phase" later in
training *reduces* $I(X; T_\ell)$, actively discarding information not needed for the task.
Plotting $I(X;T_\ell)$ against $I(T_\ell;Y)$ across training epochs produces an **information
plane**: both quantities tend to rise together early on; a compression phase, if it happens,
would show $I(X;T_\ell)$ decreasing while $I(T_\ell;Y)$ stays high. This specific claim has been
contested — Saxe et al. (2018) showed it depends heavily on activation function and how mutual
information is *estimated* from continuous activations. We measure this directly below and
report the honest result rather than overclaiming a clean compression phase.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Sweeping the compression-prediction tradeoff parameter $\beta$ from small to large traces out
the information bottleneck curve: the best achievable $I(T;Y)$ for every compression budget
$I(X;T)$, rising from total collapse to the $I(X;Y)$ ceiling.

In [ ]:
def mutual_info_bits(joint):
    p_row = joint.sum(axis=1, keepdims=True)
    p_col = joint.sum(axis=0, keepdims=True)
    support = joint > 0
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = joint / (p_row * p_col)
    return np.sum(joint[support] * np.log2(ratio[support]))

def blahut_arimoto_ib(p_xy, n_t, beta, n_iters=300, rng=None, eps=1e-12):
    """Alternately update the stochastic encoder p(t|x) and its induced marginals p(t), p(y|t)
    until convergence, for a FIXED beta. Larger beta favors I(T;Y) more; smaller beta compresses harder."""
    n_x, n_y = p_xy.shape
    p_x = p_xy.sum(axis=1)
    p_y_given_x = p_xy / p_x[:, None]
    rng = rng or np.random.default_rng(0)
    p_t_given_x = rng.dirichlet(np.ones(n_t), size=n_x)   # random row-stochastic init

    for _ in range(n_iters):
        p_t = p_t_given_x.T @ p_x
        p_x_given_t = (p_t_given_x * p_x[:, None]) / np.clip(p_t[None, :], eps, None)
        p_y_given_t = p_x_given_t.T @ p_y_given_x
        # p(t|x) proportional to p(t) * exp(-beta * D_KL(p(y|x) || p(y|t))) -- the IB fixed-point update
        log_py_x = np.log(np.clip(p_y_given_x, eps, None))
        log_py_t = np.log(np.clip(p_y_given_t, eps, None))
        kl_x_t = np.einsum("xy,xy->x", p_y_given_x, log_py_x)[:, None] - p_y_given_x @ log_py_t.T
        log_unnormalized = np.log(np.clip(p_t, eps, None))[None, :] - beta * kl_x_t
        log_unnormalized -= log_unnormalized.max(axis=1, keepdims=True)   # stability before exp
        p_t_given_x = np.exp(log_unnormalized)
        p_t_given_x /= p_t_given_x.sum(axis=1, keepdims=True)

    p_t = p_t_given_x.T @ p_x
    p_x_given_t = (p_t_given_x * p_x[:, None]) / np.clip(p_t[None, :], eps, None)
    p_y_given_t = p_x_given_t.T @ p_y_given_x
    I_XT = mutual_info_bits(p_t_given_x * p_x[:, None])
    I_TY = mutual_info_bits(p_t[:, None] * p_y_given_t)
    return I_XT, I_TY

# A controlled joint distribution: 12 values of X, secretly grouped into 3 clusters that
# determine Y (3 classes) noisily -- we KNOW in advance that the "right" compression is
# collapsing X down to its cluster identity, which is exactly what a small enough T should find
cluster_of_x = np.repeat(np.arange(3), 4)
p_x = np.full(12, 1 / 12)
p_y_given_cluster = np.array([[0.8, 0.1, 0.1], [0.1, 0.8, 0.1], [0.1, 0.1, 0.8]])
p_y_given_x = p_y_given_cluster[cluster_of_x]
p_xy = p_x[:, None] * p_y_given_x
I_XY_ceiling = mutual_info_bits(p_xy)

betas = np.concatenate([np.linspace(0.1, 2.9, 8), np.linspace(3.0, 10.0, 15)])
curve = []
for beta in betas:
    best = max((blahut_arimoto_ib(p_xy, n_t=3, beta=beta, rng=np.random.default_rng(seed))
                for seed in range(3)), key=lambda pair: pair[1])
    curve.append(best)
curve = np.array(curve)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.plot(curve[:, 0], curve[:, 1], marker="o", color="#4C72B0", lw=2, markersize=4,
        label="IB curve (best I(T;Y) per compression level)")
ax.axhline(I_XY_ceiling, color="#DD8452", ls="--", lw=1.5, label=f"I(X;Y) ceiling = {I_XY_ceiling:.3f} bits")
ax.set_xlabel("I(X;T)  [bits]  -- compression cost")
ax.set_ylabel("I(T;Y)  [bits]  -- predictive value")
ax.set_title("Information bottleneck curve:\ncompression-prediction tradeoff as beta sweeps")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"At beta->0: representation collapses, I(X;T) and I(T;Y) both -> 0.")
print(f"At beta->infinity: I(T;Y) saturates at the I(X;Y) ceiling = {I_XY_ceiling:.4f} bits, achieved once")
print(f"I(X;T) reaches only {curve[-1,0]:.4f} bits -- far below the full H(X)={np.log2(12):.3f} bits of the raw")
print(f"input, since T only needs to recover cluster identity, not every detail of X.")

## 🐍 Implementation from Scratch

We implement the Blahut-Arimoto algorithm for the information bottleneck: alternately update a
stochastic encoder $p(t|x)$ and its induced marginals until convergence, for a fixed $\beta$.
We then confirm the two boundary regimes the theory predicts -- full collapse as $\beta\to 0$,
and the $I(X;Y)$ ceiling as $\beta\to\infty$.

In [ ]:
# --- Boundary regime checks: the two extremes the IB theory explicitly predicts ---
print("Boundary regime 1: beta -> 0 should fully collapse the representation (I(X;T), I(T;Y) -> 0)")
for beta in [0.01, 0.1, 0.5, 1.0]:
    I_XT, I_TY = blahut_arimoto_ib(p_xy, n_t=3, beta=beta, n_iters=300, rng=np.random.default_rng(0))
    print(f"  beta={beta:5.2f}   I(X;T)={I_XT:.5f}   I(T;Y)={I_TY:.5f}")

print("\nBoundary regime 2: beta -> infinity should let T recover the FULL I(X;Y) ceiling")
for beta in [10.0, 30.0, 100.0, 500.0]:
    I_XT, I_TY = blahut_arimoto_ib(p_xy, n_t=3, beta=beta, n_iters=300, rng=np.random.default_rng(0))
    print(f"  beta={beta:6.1f}   I(X;T)={I_XT:.5f}   I(T;Y)={I_TY:.5f}   (ceiling I(X;Y)={I_XY_ceiling:.5f})")

# --- Confirm the encoder found by Blahut-Arimoto at a large beta actually recovers the
# TRUE cluster structure we built into the problem, not just matching numbers by coincidence ---
n_x_local, n_y_local = p_xy.shape
p_x_local = p_xy.sum(axis=1)
p_y_given_x_local = p_xy / p_x_local[:, None]
rng = np.random.default_rng(0)
p_t_given_x = rng.dirichlet(np.ones(3), size=n_x_local)
beta_large = 50.0
for _ in range(400):
    p_t = p_t_given_x.T @ p_x_local
    p_x_given_t = (p_t_given_x * p_x_local[:, None]) / np.clip(p_t[None, :], 1e-12, None)
    p_y_given_t = p_x_given_t.T @ p_y_given_x_local
    log_py_x = np.log(np.clip(p_y_given_x_local, 1e-12, None))
    log_py_t = np.log(np.clip(p_y_given_t, 1e-12, None))
    kl_x_t = np.einsum("xy,xy->x", p_y_given_x_local, log_py_x)[:, None] - p_y_given_x_local @ log_py_t.T
    log_unnorm = np.log(np.clip(p_t, 1e-12, None))[None, :] - beta_large * kl_x_t
    log_unnorm -= log_unnorm.max(axis=1, keepdims=True)
    p_t_given_x = np.exp(log_unnorm)
    p_t_given_x /= p_t_given_x.sum(axis=1, keepdims=True)

hard_assignment = p_t_given_x.argmax(axis=1)
print(f"\nLearned encoder's hard cluster assignment for each of the 12 x-values:\n  {hard_assignment}")
print(f"TRUE cluster structure built into the problem:\n  {cluster_of_x}")
print("(cluster labels may be permuted -- what matters is whether x's grouped in the TRUE clusters")
print(" also land in the SAME learned cluster together, which is the real claim being checked)")

## 🤖 Why This Matters for AI

The information bottleneck offers one influential lens on why deep networks generalize instead
of just memorizing: layers are hypothesized to progressively compress the input, discarding
task-irrelevant information while retaining what predicts $Y$. This isn't just theoretical
vocabulary — it directly motivates techniques like the **VIB (Variational Information
Bottleneck)**, which adds an explicit $I(X;T)$ penalty to a network's training objective to force
representations toward exactly this compression-prediction tradeoff, and connects to why
aggressively compressed bottleneck layers (autoencoders, VAEs) can still support strong
downstream task performance. Below, we train a small classifier and track $I(X;T_\ell)$ and
$I(T_\ell;Y)$ for its hidden layer across training epochs, plotting the resulting trajectory on
an information plane exactly as Tishby & Zaslavsky proposed — while being explicit about what the
result does and does not establish.

In [ ]:
import torch
import torch.nn as nn
from collections import Counter

# Train a tiny MLP on a task that provably REQUIRES combining all inputs (parity of 4 bits --
# no single bit, or even any 3 bits, is enough; you need all 4) and track (I(X;T), I(T;Y)) for
# its hidden layer across training, reproducing the "information plane" from Tishby & Zaslavsky.
torch.manual_seed(0)
n_bits = 4
X_bits = np.array([[int(b) for b in format(i, f"0{n_bits}b")] for i in range(2 ** n_bits)], dtype=np.float32)
Y_labels = (X_bits.sum(axis=1) % 2).astype(int)   # parity task
X_t, Y_t = torch.tensor(X_bits), torch.tensor(Y_labels)

def joint_mi_bits(codes_a, codes_b):
    # Exact discrete mutual information from empirical joint counts -- valid here because we
    # enumerate ALL 16 possible inputs exactly, so there's no sampling noise to worry about.
    n = len(codes_a)
    joint_counts, a_counts, b_counts = Counter(zip(codes_a, codes_b)), Counter(codes_a), Counter(codes_b)
    return sum((c / n) * np.log2((c / n) / ((a_counts[a] / n) * (b_counts[b] / n)))
               for (a, b), c in joint_counts.items())

class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.fc2 = nn.Linear(8, 2)
    def forward(self, x):
        hidden = torch.tanh(self.fc1(x))
        return self.fc2(hidden), hidden

torch.manual_seed(0)
model = TinyMLP()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = nn.CrossEntropyLoss()
x_codes = [tuple(row) for row in X_bits.astype(int)]
y_codes = list(Y_labels)

def quantize_hidden(hidden_tensor, n_bins=2):
    # discretize each neuron's tanh output (range [-1,1]) into n_bins even bins, then treat
    # the tuple of per-neuron bin indices as this sample's representation "code" for T
    edges = np.linspace(-1, 1, n_bins + 1)[1:-1]
    bin_idx = np.digitize(hidden_tensor.detach().numpy(), edges)
    return [tuple(row) for row in bin_idx]

checkpoints = [0, 20, 50, 100, 200, 300, 1000, 2000]
epoch = 0
print(f"{'epoch':>6} | {'I(X;T)':>8} | {'I(T;Y)':>8} | {'accuracy':>9}")
for target in checkpoints:
    while epoch < target:
        optimizer.zero_grad()
        logits, _ = model(X_t)
        loss = loss_fn(logits, Y_t)
        loss.backward()
        optimizer.step()
        epoch += 1
    with torch.no_grad():
        logits, hidden = model(X_t)
        accuracy = (logits.argmax(dim=1) == Y_t).float().mean().item()
    codes = quantize_hidden(hidden)
    I_XT, I_TY = joint_mi_bits(x_codes, codes), joint_mi_bits(codes, y_codes)
    print(f"{target:6d} | {I_XT:8.4f} | {I_TY:8.4f} | {accuracy:9.3f}")

H_X, I_XY = np.log2(16), joint_mi_bits(x_codes, y_codes)
print(f"\nH(X)=log2(16)={H_X:.4f} bits (I(X;T) ceiling); I(X;Y)={I_XY:.4f} bits (I(T;Y) ceiling)")
print("I(T;Y) rises monotonically to its ceiling -- the 'fitting' phase every account of the")
print("information plane agrees on. But I(X;T) ALSO rises straight to its ceiling and stays there:")
print("no visible compression phase in this run. That's an honest, informative result -- Tishby &")
print("Zaslavsky's compression phase is NOT a universal law. Saxe et al. (2018) showed it appears")
print("or vanishes depending on activation function and quantization, exactly the sensitivity")
print("this notebook's theory section flagged before you saw a single number.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Consider a simplified version of this notebook's clustered construction: 4 values of $X$,
   uniformly distributed, split into 2 clusters of 2 (cluster 0 = $\{x_1,x_2\}$, cluster 1 =
   $\{x_3,x_4\}$), with $P(Y{=}0\mid\text{cluster }0)=0.9$ and $P(Y{=}1\mid\text{cluster
   }1)=0.9$. By hand, compute $H(Y)$, $H(Y\mid X{=}x_1)$ (using $x_1$'s cluster-0 conditional),
   and use $I(X;Y)=H(Y)-H(Y|X)$ to get $I(X;Y)$ (note $H(Y|X{=}x)$ is identical for every $x$ by
   this construction, so $H(Y|X)$ — the *average* — equals that shared value). As a sanity check,
   confirm your result is non-negative, then verify with `mutual_info_bits` on the full joint PMF.

<details>
<summary>💡 Solution</summary>

By symmetry (2 values in each cluster, clusters symmetric under swapping labels), $P(Y{=}0) =
P(Y{=}1) = 0.5$, so $H(Y)=1$ bit. For any single $x$, $H(Y|X{=}x) = H(\text{Bernoulli}(0.9)) =
-0.9\log_2 0.9 - 0.1\log_2 0.1 \approx 0.4690$ bits — the same value for every $x$, so the
average $H(Y|X)$ also equals $0.4690$ bits. Then $I(X;Y) = H(Y)-H(Y|X) \approx 1 - 0.4690 =
0.5310$ bits, which is positive as required. Building the full $4\times 2$ joint PMF and calling
`mutual_info_bits` on it returns the same `0.5310` bits, confirming the by-hand computation.

</details>

### 💭 UNDERSTAND
2. Explain in your own words why $I(T;Y) \le I(X;Y)$ must always hold for any representation
   $T$ built only from $X$ (i.e. $Y\to X\to T$ forms a Markov chain in the sense that $T$'s
   only access to information about $Y$ is mediated through $X$). Which theorem from `05.04`
   guarantees this? Verify it numerically using two or three of the `(beta, I(X;T), I(T;Y))`
   triples already computed in this notebook's Visual Intuition section.

<details>
<summary>💡 Solution</summary>

This is exactly the data processing inequality from `05.04`, applied to the chain $Y\to X\to T$:
since $T$ is built entirely from $X$ (the encoder $p(t|x)$ never sees $Y$ directly), $T$ cannot
know anything about $Y$ that wasn't already recoverable from $X$ — every bit of information
$I(T;Y)$ carries about $Y$ had to pass *through* $X$ first, and information can only be lost or
preserved along that path, never manufactured from nothing. Checking the notebook's own computed
triples confirms this: at every $\beta$ swept in the Visual Intuition section, $I(T;Y)$ never
exceeds the $I(X;Y)=0.663$-bit ceiling, and the largest values (at large $\beta$) approach that
ceiling but never cross it.

</details>

### ✏️ DERIVE
3. Derive the identity $I(T;Y) = I(X;Y) - \sum_{x,t}P(x)P(t|x)\,D_{KL}\big(P(y|x)\,\|\,P(y|t)\big)$,
   which says the "predictive information lost" by compressing $X$ into $T$ is exactly an
   average KL-divergence penalty for how much each $T{=}t$'s conditional $P(y|t)$ differs from
   the finer-grained $P(y|x)$ it's summarizing. (Hint: start from
   $I(X;Y)-I(T;Y) = \mathbb{E}\big[\log_2\frac{P(y|x)}{P(y)}\big] - \mathbb{E}\big[\log_2\frac{P(y|t)}{P(y)}\big]$
   using the $I(A;B)=\mathbb{E}[\log_2\frac{P(b|a)}{P(b)}]$ form of mutual information, taken
   over the joint distribution of $(X,Y,T)$.)

<details>
<summary>💡 Solution outline</summary>

Using $I(A;B)=\mathbb{E}_{a,b}\big[\log_2\frac{P(b|a)}{P(b)}\big]$, write both mutual
informations as expectations over the *same* joint $(X,Y,T)$ distribution (valid since $T$ is a
stochastic function of $X$, so $(X,Y,T)\sim P(x)P(y|x)P(t|x)$ jointly):
$$I(X;Y)-I(T;Y) = \mathbb{E}_{x,y,t}\Big[\log_2\frac{P(y|x)}{P(y)} - \log_2\frac{P(y|t)}{P(y)}\Big] = \mathbb{E}_{x,y,t}\Big[\log_2\frac{P(y|x)}{P(y|t)}\Big]$$
(the $\log_2 P(y)$ terms cancel exactly). Now expand the expectation over $(x,y,t)$ by first
summing over $y$ for each fixed $(x,t)$ pair, using the joint's factorization
$P(x,y,t)=P(x)P(t|x)P(y|x)$ (since $Y\perp T \mid X$, another consequence of $T$ depending only
on $X$):
$$\mathbb{E}_{x,y,t}\Big[\log_2\frac{P(y|x)}{P(y|t)}\Big] = \sum_{x,t}P(x)P(t|x)\sum_y P(y|x)\log_2\frac{P(y|x)}{P(y|t)} = \sum_{x,t}P(x)P(t|x)\,D_{KL}\big(P(y|x)\|P(y|t)\big)$$
(the inner sum over $y$ is exactly the definition of $D_{KL}(P(y|x)\|P(y|t))$). Rearranging
gives the claimed identity. Since $D_{KL}\ge 0$ termwise (Gibbs' inequality, `05.02`), this also
gives an independent derivation of $I(T;Y)\le I(X;Y)$ (Exercise 2), now with an explicit,
non-negative "information lost" term rather than just an inequality. A numerical check
constructing a random $(X,Y,T)$ joint and computing both sides confirms the identity to
floating-point precision.

</details>

### 🐍 IMPLEMENT
4. Rerun `blahut_arimoto_ib` on this notebook's clustered `p_xy` with `n_t=6` instead of `n_t=3`
   (giving the encoder twice as many representation categories as there are true clusters), for
   several values of `beta`. Compare the resulting $(I(X;T), I(T;Y))$ pairs to the `n_t=3` curve
   already in the notebook. Does the extra capacity in $T$ let it exceed the $n_t{=}3$ curve at
   the same $\beta$?

<details>
<summary>✅ How to know you're right</summary>

At every tested $\beta$, the best `n_t=6` result (across a few random restarts, since Blahut-
Arimoto only finds a local optimum) should match the `n_t=3` result closely — both $I(X;T)$ and
$I(T;Y)$ should agree to within a small numerical tolerance (a representative run: at
$\beta{=}10$, both settings converge to $I(X;T)\approx 1.585$, $I(T;Y)\approx 0.663$, the
exact $I(X;Y)$ ceiling). The extra capacity does *not* help, because the true structure in this
problem only requires 3 distinguishable representation categories (matching the 3 real
clusters) — a correctly-converged encoder with `n_t=6` simply leaves the extra categories
redundant (effectively merging or barely using them) rather than finding a genuinely better
solution. If your `n_t=6` result meaningfully *exceeds* the `n_t=3` curve at the same $\beta$,
that's a sign the `n_t=3` run got stuck in a worse local optimum on that particular random
restart, not that more categories are fundamentally more powerful here — rerun both with more
random restarts (`rng` seeds) and keep the best result at each `n_t`.

</details>

### 🤖 APPLY
5. A model architect wants to choose a bottleneck layer's capacity for a task shaped like this
   notebook's clustered example. They specify a design target: retain at least 90% of the
   $I(X;Y)$ ceiling while paying as little $I(X;T)$ "compression cost" as possible. Using the
   `(beta, I(X;T), I(T;Y))` triples from this notebook's own $\beta$-sweep (or a finer sweep of
   your own), find the smallest $\beta$ (and corresponding $I(X;T)$) that meets the 90% target.

<details>
<summary>✅ What you should observe</summary>

The $90\%$ target is $0.9\times 0.663 \approx 0.597$ bits of $I(T;Y)$. Scanning the notebook's
own beta sweep (or a finer one), the target is first met somewhere in the neighborhood of
$\beta\approx 3$–$3.5$, at which point $I(X;T)$ has reached roughly $1.3$–$1.5$ bits — notably
*less* than the $I(X;T)\approx 1.58$ bits eventually paid to reach $100\%$ of the ceiling at
large $\beta$. This is the compression-prediction tradeoff made concrete as a design decision:
capturing the "easy" 90% of predictive value costs meaningfully less representational capacity
than squeezing out the last 10%, which is exactly the kind of diminishing-returns curve that
motivates choosing a bottleneck width by a target performance level rather than "as large as
possible."

</details>

### 🚀 CHALLENGE
6. In the AI section, replace `torch.tanh` with `torch.relu` as the hidden layer's activation,
   retrain, and recompute the same `(I(X;T), I(T;Y))` trajectory. Does a compression phase
   appear, disappear, or look about the same? Write 2-3 sentences connecting your empirical
   finding to why the "compression phase" claim has been such a contested part of deep learning
   theory rather than a settled fact.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer finds that neither activation shows a clean compression phase in this setup —
both `tanh` and `relu` runs show $I(X;T)$ rising (or staying roughly flat/noisy) rather than
decreasing after the fitting phase, though the two trajectories differ in their details (a
representative comparison: `tanh` reaches near-ceiling $I(T;Y)$ and near-$\log_2(16)=4$-bit
$I(X;T)$ by around epoch 200 and holds there, while `relu` converges more slowly, to a lower
final accuracy and a lower $I(X;T)$, reflecting `relu`'s very different activation range and the
quantization scheme's sensitivity to that range). A strong answer explicitly notes that this
sensitivity — the measured $I(X;T)$ trajectory changing noticeably with the *activation
function*, and being highly dependent on binning/quantization choices for a continuous
activation — is itself the empirical core of Saxe et al.'s (2018) critique: the originally
reported "compression phase" may have been substantially an artifact of the specific binning
estimator and the saturating `tanh` nonlinearity used in the original Tishby & Zaslavsky
experiments, rather than a universal property of deep network training.

</details>

---

## 📚 Further Reading
- Tishby, Pereira & Bialek, ["The Information Bottleneck Method"](https://arxiv.org/abs/physics/0004057) (1999)
- Tishby & Zaslavsky, ["Deep Learning and the Information Bottleneck Principle"](https://arxiv.org/abs/1503.02406) (2015)
- Saxe et al., ["On the Information Bottleneck Theory of Deep Learning"](https://arxiv.org/abs/1810.05728) (2018) — the critical follow-up

---

*Next notebook: [Coding Theory Basics](06_coding_theory_basics.ipynb)*